# Anatomy of a simulation

A `Simulation` ties together five objects (the legacy six minus the
post-processor; memory retention is now a constructor kwarg):

- **Founder haplotypes** — generation-0 haplotypes
  (`xftsim.struct.DenseHaplotypeArray` or `GraphHaplotypeOperator`).
  Built from real data or simulated.
- **Recombination maps** (`xftsim.reproduce.RecombinationMap`) —
  per-locus recombination probabilities used during meiosis.
- **Phenogenetic architectures** (`xftsim.arch.Architecture`) —
  a DAG of `ArchNode`s, almost always built from a lavaan-style
  formula string by the `parser` module.
- **Mating regimes** (`xftsim.mate.RandomMating`,
  `LinearAssortativeMating`, `GeneralAssortativeMating`,
  `BatchedMating`) — decide who pairs with whom and how many
  offspring each pair produces.
- **Statistics** (`xftsim.stats.Statistic` subclasses) — anything you
  want to compute and stash each generation. Optional.

Plus two cross-cutting concepts:

- **Filters** (`xftsim.filters.Filter` subclasses) — named views over
  the current generation (`TrioFilter`, `SibPairFilter`,
  `UnrelatedFilter`, `AscertainmentFilter`, `SubsampleFilter`).
  Statistics like `MatingStatistics` and `ParentOffspringRegression`
  consume these views.
- **Memory retention** — the legacy `xftsim.proc.LimitMemory`
  post-processor has been removed. Pass
  `retain_haplotypes=N` / `retain_phenotypes=N` to `Simulation`
  to control how many past generations are kept.

We'll go through each of these in detail in the following sections.

The minimal end-to-end pattern looks like this:


In [ ]:
import xftsim as xft
import numpy as np

# 1. Founders
hap = xft.founders.founder_haplotypes_uniform_AFs(n=800, m=200)

# 2. Architecture (formula DSL)
eff = xft.effect.AdditiveEffects.from_h2(h2=0.5, m=200, seed=1)
arch = xft.arch.Architecture(
    formula='''
    Y.G ~ genetic(eff)
    Y.E ~ noise(0.5)
    Y   ~ Y.G + Y.E
    ''',
    effects={'eff': eff},
)

# 3. Recombination map
rmap = xft.reproduce.RecombinationMap.from_haplotypes(hap, p=0.1)

# 4. Mating regime
mating = xft.mate.RandomMating(offspring_per_pair=2)

# 5. Stats
stats_list = [xft.stats.SampleStatistics(),
              xft.stats.HasemanElstonEstimator()]

# Tie it all together
sim = xft.sim.Simulation(
    founder_haplotypes=hap,
    architecture=arch,
    recombination_map=rmap,
    mating_regime=mating,
    statistics=stats_list,
    retain_haplotypes=1,
    retain_phenotypes=2,
    seed=42,
)
sim.run(n_generations=3)


After `sim.run(n)`, results for each generation live in `sim.results`
(a list of `GenerationResult` dataclasses, one per completed
generation):


In [ ]:
for r in sim.results:
    print(f'gen {r.generation}: {list(r.statistics.keys())}')


There is also a more readable end-to-end example in
[`xftsim/quickstart.py`](https://github.com/border-lab/xftsim/blob/v0.9alpha/xftsim/quickstart.py).
That module also documents the legacy → v0.9 API translations at the
top of the file, which is a handy reference if you're porting code
from the old user guide.
